1. parse the pdf file
2. apply chunking & embedding
3. metadata extraction

### **PDF Parsing**

In [81]:
import os
import re
from typing import List, Optional
from uuid import uuid4

from rich import print as rptint
from dotenv import load_dotenv
import numpy as np
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_mongodb.index import create_fulltext_search_index
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers.hybrid_search import MongoDBAtlasHybridSearchRetriever



load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI')
SOURCE_PDF_DIR = '../data/海外旅行不便險條款.pdf'

### Mongodb initalize

In [82]:
client = MongoClient(MONGODB_URI)
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
dim = len(embeddings.embed_query('foo'))

DB_NAME = 'agent_db'
COLLECTION_NAME = 'cxi-travel-ins-collection'
ATLAS_VECTOR_SEARCH_INDEX_NAME = "cxi-travel-ins-qa-vector-index"

# COLLECTION_NAME = 'cxi-travel-ins-collection-2'
# ATLAS_VECTOR_SEARCH_INDEX_NAME = "cxi-travel-ins-qa-vector-index-2"
MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

In [ ]:
vector_store = MongoDBAtlasVectorSearch(
    collection=MONGODB_COLLECTION,
    embedding=embeddings,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    relevance_score_fn="cosine",
    embedding_key="embedding",
    text_key="text"
)

# create mongodb fulltext search index with lucene.smartcn analyzer
fulltext_search_definition = {
    "mappings": {
        "dynamic": False,
        "fields": {
            "text": {
                "type": "string",
                "analyzer": "lucene.chinese"
            }
        }
    }
}

vector_search_definition = {
    "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": dim,
                "similarity": "cosine"
            },
            {
                "type": "filter", 
                "path": "anchor_page"
            },
            {
                "type": "filter", 
                "path": "article"
            }
        ]
}

# create mongodb vector search index
# vector_store.create_vector_search_index(dimensions=dim) # create index through langchain
vector_search_index_model = SearchIndexModel(
    vector_search_definition,
    name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    type="vectorSearch")

# create mongodb (fulltext) search index
search_index_model = SearchIndexModel(
   definition=fulltext_search_definition,
   name="search_index"
)
MONGODB_COLLECTION.create_search_index(model=search_index_model)

'search_index'

In [57]:
ARTICLE_RE = re.compile(
    r"(?m)^\s*第(?:[零一二三四五六七八九十百千萬]+|\d+)條(?:之\d+)?"
)

def merge_pages_by_article(page_docs: List[Document]) -> List[Document]:
    """
    根據條款分界（如「第X條」）將連續的文件頁面內容合併。

    此函數會依據 ARTICLE_RE 規則，將屬於同一條款的文本（即使跨越多個頁面）合併為一個 Document 實例。
    每一個合併後的 Document 會包含：
      - 條款完整內容（可能橫跨多頁）
      - 條款編號
      - 所有涉及的頁碼
      - 條款在原始文件中的起始頁（anchor_page）

    TODO: 如果條款內容延伸到文件最後，附錄或附件內容可能會被計入最後一條條款。

    Args:
        page_docs (List[Document]): 按序排列的每頁 Document 物件。

    Returns:
        List[Document]: 依據條款合併後的 Document 清單。
    """
    
    merged_docs: List[Document] = []
    current_text_parts: List[str] = []
    current_pages: List[int] = []
    current_article: Optional[str] = None
    anchor_page: Optional[int] = None
    
    def add_page(page: Optional[int] = None):
        if page is not None and page not in current_pages:
            current_pages.append(page)
    
    def flush_current():
        nonlocal current_text_parts, current_pages, anchor_page, current_article
        if not current_text_parts:
            return
        
        merged_text = "\n".join(current_text_parts).strip()
        if merged_text:
            merged_docs.append(
                Document(
                    page_content=merged_text,
                    metadata={
                        "anchor_page": anchor_page,
                        "pages": current_pages.copy(),
                        "article": current_article,
                    }
                )
            )
            
        current_text_parts = []
        current_pages = []
        anchor_page = None
        current_article = None
        
    for doc in page_docs:
        text = (doc.page_content or "").strip()
        page = doc.metadata.get('page')
        
        if not text:
            continue
        
        groups = [g for g in ARTICLE_RE.finditer(text)] 
        starts = [group.start() for group in groups] # 條款開始字數 index
        group = [g.group(0) for g in groups] # 條款
        
        if not starts:
            # 本頁無條款
            if not current_text_parts:
                # 代表一開始就遇到無條款頁
                anchor_page = page
                
            current_text_parts.append(text)
            if page not in current_pages:
                current_pages.append(page)
            continue
        
        # 處理前一條款
        if starts[0] > 0:
            previous_article_text = text[:starts[0]].strip()
            if previous_article_text:
                if not current_text_parts:
                    anchor_page = page
                current_text_parts.append(previous_article_text)
                add_page(page)
                
                
        flush_current()
        
        starts.append(len(text))
        page_segments = [
            text[starts[i]:starts[i + 1]].strip()
            for i in range(len(starts) - 1)
            if text[starts[i]:starts[i + 1]].strip()
        ]
        
        for idx, seg in enumerate(page_segments):
            anchor_page = page
            current_text_parts = [seg]
            current_pages = [page]
            current_article = group[idx]
            
            if idx < len(page_segments) - 1:
                flush_current()

    flush_current()
    
    return merged_docs
    

In [50]:
file_loader = PyMuPDFLoader(SOURCE_PDF_DIR, mode='page', extract_tables='markdown')
raw_docs = file_loader.load()
article_docs = merge_pages_by_article(raw_docs)

In [ ]:
vector_store.add_documents(article_docs)

In [ ]:
retriever = MongoDBAtlasHybridSearchRetriever(
    vectorstore=vector_store,
    search_index_name="search_index",
    top_k=5,
    fulltext_penalty=50,
    vector_penalty=50,
    post_filter=[
        {
            '$project': {
                'embedding': 0,
            }
        }
    ]
)